# Phase 4: Explainable AI (XAI) for Clinical Text Classification
**Dataset:** Medical Transcriptions (MTSamples)  
**Best Model:** BioBERT + Synonym Swap (39.11%)  
**Goal:** Explain model predictions using LIME

### What this notebook does:
1. Train the best model (BioBERT + Synonym Swap)
2. **LIME** - Local Interpretable Model-agnostic Explanations for individual predictions
3. **Global Feature Importance** - Aggregated LIME analysis
4. Analyze which words drive predictions for each medical specialty

**Requires GPU**

## 1. Install & Import Libraries

In [15]:
!pip install transformers datasets accelerate lime shap -q

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import re
import random
import torch
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

import lime
import lime.lime_text

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

random.seed(42)
np.random.seed(42)

Device: cuda
GPU: Tesla T4


## 2. Load Data & Apply Synonym Swap

In [16]:
df = pd.read_csv("/kaggle/input/datasets/mstarefin/medicaltranscriptions-4/mtsamples.csv")
df = df.dropna(subset=["transcription"])

specialty_counts = df["medical_specialty"].value_counts()
MIN_SAMPLES = 50
valid_specialties = specialty_counts[specialty_counts >= MIN_SAMPLES].index.tolist()
df_filtered = df[df["medical_specialty"].isin(valid_specialties)].copy()

le = LabelEncoder()
df_filtered["label"] = le.fit_transform(df_filtered["medical_specialty"])
num_labels = len(le.classes_)

MEDICAL_SYNONYMS = {
    "heart attack": ["myocardial infarction", "MI", "cardiac event"],
    "myocardial infarction": ["heart attack", "MI", "cardiac event"],
    "high blood pressure": ["hypertension", "elevated BP", "HTN"],
    "hypertension": ["high blood pressure", "elevated BP", "HTN"],
    "diabetes": ["diabetes mellitus", "DM", "diabetic condition"],
    "stroke": ["cerebrovascular accident", "CVA", "brain attack"],
    "kidney": ["renal"],
    "renal": ["kidney"],
    "liver": ["hepatic"],
    "hepatic": ["liver"],
    "lungs": ["pulmonary"],
    "pulmonary": ["lungs", "lung"],
    "stomach": ["gastric", "abdominal"],
    "gastric": ["stomach"],
    "cancer": ["malignancy", "neoplasm", "carcinoma"],
    "malignancy": ["cancer", "neoplasm"],
    "pain": ["discomfort", "tenderness", "soreness"],
    "swelling": ["edema", "inflammation"],
    "edema": ["swelling"],
    "bleeding": ["hemorrhage", "hemorrhaging"],
    "hemorrhage": ["bleeding"],
    "fracture": ["break", "broken bone"],
    "infection": ["sepsis", "infectious process"],
    "surgery": ["surgical procedure", "operation", "operative procedure"],
    "headache": ["cephalgia", "head pain"],
    "fever": ["pyrexia", "febrile", "elevated temperature"],
    "nausea": ["queasiness", "feeling sick"],
    "vomiting": ["emesis"],
    "shortness of breath": ["dyspnea", "SOB", "breathlessness"],
    "dyspnea": ["shortness of breath", "SOB", "breathlessness"],
    "chest pain": ["thoracic pain", "chest discomfort"],
}

def synonym_swap(text, swap_prob=0.3):
    augmented = text
    text_lower = text.lower()
    for term, synonyms in MEDICAL_SYNONYMS.items():
        if term in text_lower and random.random() < swap_prob:
            synonym = random.choice(synonyms)
            pattern = re.compile(re.escape(term), re.IGNORECASE)
            augmented = pattern.sub(synonym, augmented, count=1)
    return augmented

def light_clean(text):
    return re.sub(r"\s+", " ", text).strip()

df_filtered["clean_text"] = df_filtered["transcription"].apply(synonym_swap).apply(light_clean)

print("Samples:", len(df_filtered))
print("Classes:", num_labels)

Samples: 4647
Classes: 22


## 3. Train Best Model (BioBERT + Synonym Swap)

In [17]:
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"
MAX_LENGTH = 512

X = df_filtered["clean_text"].values
y = df_filtered["label"].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

train_ds = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
val_ds = Dataset.from_dict({"text": X_val.tolist(), "label": y_val.tolist()})
test_ds = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, ignore_mismatched_sizes=True
)

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": accuracy_score(labels, np.argmax(logits, axis=-1))}

train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)
test_tok = test_ds.map(tokenize_fn, batched=True)

training_args = TrainingArguments(
    output_dir="./results_xai",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    fp16=True if device == "cuda" else False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

print("Training BioBERT + Synonym Swap...")
trainer.train()

predictions = trainer.predict(test_tok)
preds = np.argmax(predictions.predictions, axis=-1)
test_acc = accuracy_score(y_test, preds)
print()
print("Test Accuracy: {:.2f}%".format(test_acc * 100))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

Map:   0%|          | 0/3252 [00:00<?, ? examples/s]

Map:   0%|          | 0/697 [00:00<?, ? examples/s]

Map:   0%|          | 0/698 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Test Accuracy: 39.40%


## 4. Create Prediction Function for LIME

In [18]:
def predict_proba(texts):
    model.eval()
    all_probs = []
    batch_size = 16
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(
            batch_texts, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)

test_text = X_test[0]
probs = predict_proba([test_text])
pred_class = le.classes_[np.argmax(probs[0])]
print("Test prediction:", pred_class)
print("Confidence: {:.2f}%".format(np.max(probs[0]) * 100))

Test prediction:  Consult - History and Phy.
Confidence: 32.27%


## 5. LIME Explainer Setup

In [19]:
explainer = lime.lime_text.LimeTextExplainer(
    class_names=le.classes_,
    split_expression=r"\W+",
    random_state=42
)
print("LIME explainer created")

LIME explainer created


## 6. LIME - Individual Prediction Explanations

In [20]:
correct_mask = preds == y_test
correct_indices = np.where(correct_mask)[0]

selected_indices = []
seen_labels = set()
for idx in correct_indices:
    label = y_test[idx]
    if label not in seen_labels and len(selected_indices) < 5:
        selected_indices.append(idx)
        seen_labels.add(label)

print("Selected {} samples for LIME explanation".format(len(selected_indices)))

Selected 5 samples for LIME explanation


In [21]:
lime_explanations = []

for i, idx in enumerate(selected_indices):
    text = X_test[idx]
    true_label = le.classes_[y_test[idx]]
    
    print("=" * 60)
    print("Sample {} - True Label: {}".format(i + 1, true_label))
    print("Text (first 200 chars): {}...".format(text[:200]))
    print()
    
    exp = explainer.explain_instance(
        text, predict_proba, num_features=15, num_samples=300, top_labels=3
    )
    lime_explanations.append(exp)
    
    pred_label_idx = y_test[idx]
    print("Top words supporting prediction ({}):".format(true_label))
    feature_list = exp.as_list(label=pred_label_idx)
    for word, weight in feature_list[:10]:
        direction = "+" if weight > 0 else "-"
        print("  {} {}: {:.4f}".format(direction, word, abs(weight)))
    print()

Sample 1 - True Label:  Consult - History and Phy.
Text (first 200 chars): IDENTIFYING DATA: , The patient is a 21-year-old Caucasian male, who attempted suicide by trying to jump from a moving car, which was being driven by his mother. Additionally, he totaled his own car e...

Top words supporting prediction ( Consult - History and Phy.):
  + injuries: 0.0073
  - basis: 0.0070
  + ILLNESS: 0.0057
  + repeated: 0.0038
  + This: 0.0032
  - psychomotor: 0.0032
  + now: 0.0028
  - was: 0.0027
  - ABC: 0.0027
  - a: 0.0023

Sample 2 - True Label:  Discharge Summary
Text (first 200 chars): DISCHARGE DIAGNOSES:,1. Advanced non-small cell lung carcinoma with left malignant pleural effusion status post chest tube insertion status post chemical pleurodesis.,2. Respiratory failure secondary ...

Top words supporting prediction ( Discharge Summary):
  + DISCHARGE: 0.0287
  + DIAGNOSES: 0.0202
  + disease: 0.0080
  + bones: 0.0077
  + hip: 0.0060
  + Acute: 0.0054
  + wife: 0.0045
  + Elevated: 0

## 7. LIME Visualizations

In [22]:
fig, axes = plt.subplots(len(selected_indices), 1, figsize=(12, 5 * len(selected_indices)))

if len(selected_indices) == 1:
    axes = [axes]

for i, (idx, exp) in enumerate(zip(selected_indices, lime_explanations)):
    true_label = le.classes_[y_test[idx]]
    pred_label_idx = y_test[idx]
    feature_list = exp.as_list(label=pred_label_idx)
    words = [f[0] for f in feature_list[:12]]
    weights = [f[1] for f in feature_list[:12]]
    colors_bar = ["#2ecc71" if w > 0 else "#e74c3c" for w in weights]
    
    axes[i].barh(range(len(words)), weights, color=colors_bar, edgecolor="black")
    axes[i].set_yticks(range(len(words)))
    axes[i].set_yticklabels(words, fontsize=10)
    axes[i].set_xlabel("Weight", fontsize=11)
    axes[i].set_title("LIME Explanation - {} (Sample {})".format(true_label, i + 1),
                      fontsize=12, fontweight="bold")
    axes[i].axvline(x=0, color="black", linewidth=0.5)
    axes[i].invert_yaxis()

plt.tight_layout()
plt.savefig("lime_explanations.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: lime_explanations.png")

Saved: lime_explanations.png


## 8. Global Feature Importance (Aggregated LIME)

In [23]:
SHAP_SAMPLE_SIZE = 50

shap_indices = []
samples_per_class = max(1, SHAP_SAMPLE_SIZE // num_labels)
for label_id in range(num_labels):
    class_indices = np.where(y_test == label_id)[0]
    if len(class_indices) > 0:
        selected = np.random.choice(class_indices, size=min(samples_per_class, len(class_indices)), replace=False)
        shap_indices.extend(selected)

shap_indices = shap_indices[:SHAP_SAMPLE_SIZE]
shap_texts = [X_test[i] for i in shap_indices]
shap_labels = [y_test[i] for i in shap_indices]

print("Analyzing {} samples...".format(len(shap_texts)))

word_importance_global = {}
word_class_importance = {cls: {} for cls in le.classes_}

for i, (text, label) in enumerate(zip(shap_texts, shap_labels)):
    if i % 10 == 0:
        print("Processing sample {} / {}...".format(i + 1, len(shap_texts)))
    try:
        exp = explainer.explain_instance(
            text, predict_proba, num_features=20, num_samples=200, top_labels=1
        )
        feature_list = exp.as_list(label=label)
        class_name = le.classes_[label]
        for word, weight in feature_list:
            abs_weight = abs(weight)
            if word in word_importance_global:
                word_importance_global[word].append(abs_weight)
            else:
                word_importance_global[word] = [abs_weight]
            if word in word_class_importance[class_name]:
                word_class_importance[class_name][word].append(weight)
            else:
                word_class_importance[class_name][word] = [weight]
    except Exception as e:
        continue

print()
print("Analysis complete! Unique words:", len(word_importance_global))

Analyzing 44 samples...
Processing sample 1 / 44...
Processing sample 11 / 44...
Processing sample 21 / 44...
Processing sample 31 / 44...
Processing sample 41 / 44...

Analysis complete! Unique words: 113


## 9. Global Feature Importance Plot

In [24]:
avg_importance = {word: np.mean(weights) for word, weights in word_importance_global.items()}
sorted_importance = sorted(avg_importance.items(), key=lambda x: x[1], reverse=True)

top_words = sorted_importance[:25]
words_list = [w[0] for w in top_words]
importance_list = [w[1] for w in top_words]

plt.figure(figsize=(12, 8))
plt.barh(range(len(words_list)), importance_list, color="steelblue", edgecolor="black")
plt.yticks(range(len(words_list)), words_list, fontsize=11)
plt.xlabel("Average Absolute Importance", fontsize=12)
plt.title("Top 25 Most Important Words (Global)", fontsize=14, fontweight="bold")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("global_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: global_feature_importance.png")

Saved: global_feature_importance.png


## 10. Top Words per Medical Specialty (FIXED)

In [25]:
# Find specialties that actually have LIME data
available_specialties = [s for s in word_class_importance if len(word_class_importance[s]) > 0]
print("Specialties with LIME data:", len(available_specialties))

# Pick top 5 by number of words analyzed
selected_specialties = sorted(available_specialties, key=lambda s: len(word_class_importance[s]), reverse=True)[:5]
print("Selected:", selected_specialties)

fig, axes = plt.subplots(len(selected_specialties), 1, figsize=(12, 4 * len(selected_specialties)))

if len(selected_specialties) == 1:
    axes = [axes]

for ax_idx, specialty in enumerate(selected_specialties):
    class_words = word_class_importance[specialty]
    avg_class_imp = {word: np.mean(weights) for word, weights in class_words.items()}
    sorted_class = sorted(avg_class_imp.items(), key=lambda x: abs(x[1]), reverse=True)
    
    top_n = sorted_class[:10]
    words_c = [w[0] for w in top_n]
    weights_c = [w[1] for w in top_n]
    colors_c = ["#2ecc71" if w > 0 else "#e74c3c" for w in weights_c]
    
    axes[ax_idx].barh(range(len(words_c)), weights_c, color=colors_c, edgecolor="black")
    axes[ax_idx].set_yticks(range(len(words_c)))
    axes[ax_idx].set_yticklabels(words_c, fontsize=10)
    axes[ax_idx].set_xlabel("Average Weight", fontsize=10)
    axes[ax_idx].set_title("Top Words for: {}".format(specialty), fontsize=12, fontweight="bold")
    axes[ax_idx].axvline(x=0, color="black", linewidth=0.5)
    axes[ax_idx].invert_yaxis()

plt.tight_layout()
plt.savefig("per_class_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: per_class_importance.png")

Specialties with LIME data: 5
Selected: [' Surgery', ' Cardiovascular / Pulmonary', ' Obstetrics / Gynecology', ' Ophthalmology', ' Orthopedic']
Saved: per_class_importance.png


## 11. Misclassification Analysis

In [26]:
wrong_mask = preds != y_test
wrong_indices = np.where(wrong_mask)[0]

print("Total misclassified: {} / {} ({:.1f}%)".format(
    len(wrong_indices), len(y_test), len(wrong_indices) / len(y_test) * 100
))
print()

for i, idx in enumerate(wrong_indices[:3]):
    text = X_test[idx]
    true_label = le.classes_[y_test[idx]]
    pred_label = le.classes_[preds[idx]]
    
    print("=" * 60)
    print("Misclassification {}".format(i + 1))
    print("True: {} --> Predicted: {}".format(true_label, pred_label))
    print("Text (first 200 chars): {}...".format(text[:200]))
    print()
    
    try:
        exp = explainer.explain_instance(
            text, predict_proba, num_features=10, num_samples=200, top_labels=2
        )
        print("Words pushing toward PREDICTED ({}):".format(pred_label))
        for word, weight in exp.as_list(label=preds[idx])[:5]:
            print("  {}: {:.4f}".format(word, weight))
        print()
        print("Words pushing toward TRUE ({}):".format(true_label))
        for word, weight in exp.as_list(label=y_test[idx])[:5]:
            print("  {}: {:.4f}".format(word, weight))
    except Exception as e:
        print("Could not generate explanation:", str(e))
    print()

Total misclassified: 423 / 698 (60.6%)

Misclassification 1
True:  Pediatrics - Neonatal --> Predicted:  Consult - History and Phy.
Text (first 200 chars): REASON FOR CONSULTATION: , New-onset seizure.,HISTORY OF PRESENT ILLNESS: , The patient is a 2-1/2-year-old female with a history of known febrile seizures, who was placed on Keppra oral solution at 1...

Words pushing toward PREDICTED ( Consult - History and Phy.):
  REASON: 0.0531
  rhythm: 0.0134
  dosing: 0.0115
  tomorrow: 0.0109
  those: 0.0061

Words pushing toward TRUE ( Pediatrics - Neonatal):
Could not generate explanation: np.int64(16)

Misclassification 2
True:  Surgery --> Predicted:  Orthopedic
Text (first 200 chars): PREOPERATIVE DIAGNOSIS:, Rotator cuff tear, right shoulder.,POSTOPERATIVE DIAGNOSES:,1. Massive rotator cuff tear, right shoulder.,2. Near complete biceps tendon tear, right shoulder.,3. Chondromalaci...

Words pushing toward PREDICTED ( Orthopedic):
  tendon: 0.0347
  instability: -0.0209
  biceps: 0.019

## 12. Most Confused Specialty Pairs

In [27]:
cm = confusion_matrix(y_test, preds)

confused_pairs = []
for i in range(len(le.classes_)):
    for j in range(len(le.classes_)):
        if i != j and cm[i][j] > 0:
            confused_pairs.append((le.classes_[i], le.classes_[j], cm[i][j]))

confused_pairs.sort(key=lambda x: x[2], reverse=True)

print("TOP 10 MOST CONFUSED SPECIALTY PAIRS")
print("=" * 60)
print("{:<35} {:<35} {}".format("True Label", "Predicted As", "Count"))
print("-" * 75)
for true, pred, count in confused_pairs[:10]:
    print("{:<35} {:<35} {}".format(true, pred, count))

TOP 10 MOST CONFUSED SPECIALTY PAIRS
True Label                          Predicted As                        Count
---------------------------------------------------------------------------
 General Medicine                    Consult - History and Phy.         28
 Surgery                             Cardiovascular / Pulmonary         24
 Surgery                             Orthopedic                         22
 Gastroenterology                    Surgery                            17
 Urology                             Surgery                            15
 Orthopedic                          Surgery                            12
 Radiology                           Cardiovascular / Pulmonary         12
 SOAP / Chart / Progress Notes       Consult - History and Phy.         12
 ENT - Otolaryngology                Surgery                            10
 Gastroenterology                    Consult - History and Phy.         10


## 13. Complete Project Summary

In [28]:
print("=" * 60)
print("PROJECT COMPLETE - ALL 4 PHASES")
print("=" * 60)
print()
print("Phase 1: TF-IDF + Traditional ML")
print("  Best: Logistic Regression - 27.63%")
print()
print("Phase 2: BERT Fine-tuning")
print("  BERT-base:     37.82%")
print("  BioBERT:       38.54%  (best)")
print("  ClinicalBERT:  38.25%")
print()
print("Phase 3: Numeric Mapping & Synonym Swap")
print("  Numeric Mapped:  38.83% (+0.29%)")
print("  Synonym Swap:    39.11% (+0.57%) (best)")
print("  Combined:        37.54% (-1.00%)")
print()
print("Phase 4: Explainable AI")
print("  LIME: Individual prediction explanations generated")
print("  Global feature importance analyzed")
print("  Misclassification analysis completed")
print()
print("Figures saved:")
print("  - lime_explanations.png")
print("  - global_feature_importance.png")
print("  - per_class_importance.png")

PROJECT COMPLETE - ALL 4 PHASES

Phase 1: TF-IDF + Traditional ML
  Best: Logistic Regression - 27.63%

Phase 2: BERT Fine-tuning
  BERT-base:     37.82%
  BioBERT:       38.54%  (best)
  ClinicalBERT:  38.25%

Phase 3: Numeric Mapping & Synonym Swap
  Numeric Mapped:  38.83% (+0.29%)
  Synonym Swap:    39.11% (+0.57%) (best)
  Combined:        37.54% (-1.00%)

Phase 4: Explainable AI
  LIME: Individual prediction explanations generated
  Global feature importance analyzed
  Misclassification analysis completed

Figures saved:
  - lime_explanations.png
  - global_feature_importance.png
  - per_class_importance.png
